# Cosmic-IB Dashboard — Data Exploration Notebook

Use this notebook to interactively inspect the data files produced by the IB pipeline
before running the Streamlit dashboard.

**Prerequisites:** Run `bash scripts/extract_data.sh` first to populate `extracted/`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = os.path.join('..', 'extracted')
print('Files in extracted/:', len(os.listdir(DATA_DIR)))

## 1. Fisher Information Results

In [ ]:
fisher = np.load(os.path.join(DATA_DIR, 'fisher_results.npz'), allow_pickle=True)
print('Keys:', list(fisher.keys()))
print(f'I(X;T) = {float(fisher["MI"]):.4f} nats')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(fisher['l'][1:], fisher['Cl'][1:], color='steelblue')
axes[0].set_xlabel('Multipole ℓ'); axes[0].set_ylabel('C_ℓ'); axes[0].set_title('Power Spectrum')

axes[1].plot(fisher['l'][1:], fisher['Fl'][1:], color='darkorange')
axes[1].set_xlabel('Multipole ℓ'); axes[1].set_ylabel('F_ℓ'); axes[1].set_title('Fisher Information')
plt.tight_layout()
plt.show()

## 2. η_IB Results Table

In [ ]:
rows = []
for fname, nside, vpix in [
    ('eta_IB_cosmic_lowres.npz', 64, 10409),
    ('eta_IB_nside16.npz',       16, 856),
    ('eta_IB_nside8.npz',         8, 280),
]:
    d = np.load(os.path.join(DATA_DIR, fname), allow_pickle=True)
    rows.append({'nside': nside, 'I_TY': float(d['I_TY']),
                 'eta_IB': float(d['eta_IB']), 'valid_pixels': vpix})

df = pd.DataFrame(rows).sort_values('nside', ascending=False)
df['pixel_area_deg2'] = (4 * 180**2 / np.pi) / (12 * df['nside']**2)
display(df)

## 3. HEALPix Maps at nside=16

In [ ]:
try:
    import healpy as hp
    density = hp.read_map(os.path.join(DATA_DIR, 'density_nside16.fits'), verbose=False)
    cluster = hp.read_map(os.path.join(DATA_DIR, 'cluster_count_nside16.fits'), verbose=False)

    fig = plt.figure(figsize=(14, 5))
    hp.mollview(density, fig=1, sub=121, title='Galaxy Density T (nside=16)', cmap='plasma')
    hp.mollview(cluster, fig=1, sub=122, title='Cluster Count Y (nside=16)', cmap='inferno')
    plt.show()
except ImportError:
    print('healpy not installed — skipping HEALPix visualisation')

## 4. ETF Bootstrap Distribution

In [ ]:
bs = np.load(os.path.join(DATA_DIR, 'etf_scores_bootstrap.npy'))
etf_full = float(np.load(os.path.join(DATA_DIR, 'etf_score_full.npy')))
phi_tda  = float(np.load(os.path.join(DATA_DIR, 'phi_tda.npy')))

print(f'Φ_TDA = {phi_tda:.2f}')
print(f'ETF full sample = {etf_full:.4f}')
print(f'ETF bootstrap   = {bs.mean():.4f} ± {bs.std():.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(bs, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(etf_full, color='orange', linewidth=2, label=f'Full sample ({etf_full:.3f})')
ax.set_xlabel('ETF score'); ax.set_ylabel('Count'); ax.set_title('Bootstrap ETF Scores (n=100)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Hyper-IB Ricci Results

In [ ]:
ricci = np.load(os.path.join(DATA_DIR, 'hyper_ib_ricci_results.npz'), allow_pickle=True)
for k in ricci.keys():
    print(f'  {k}: {float(ricci[k]):.6f}')